# 05_artists

DML: bronze_artists — Raw artist metadata.

In [ ]:
%run ../../tools/config/settings

In [ ]:
dbutils.widgets.text("run_id",         "")
dbutils.widgets.text("ingestion_date", "")
run_id         = dbutils.widgets.get("run_id")
ingestion_date = dbutils.widgets.get("ingestion_date")

In [ ]:
from pyspark.sql import functions as F

raw_path = raw_base_path("artists", ingestion_date, run_id)
raw_df   = spark.read.json(f"{raw_path}/*.json")

bronze = (
    raw_df
    .select(F.explode("artists").alias("a"))
    .select(
        F.col("a.id").alias("artist_id"),
        F.col("a.name").alias("artist_name"),
        F.col("a.genres").alias("genres"),
        F.col("a.popularity").alias("popularity"),
        F.col("a.followers.total").alias("followers_total"),
        F.to_json(F.col("a")).alias("_raw"),
    )
    .withColumn("run_id",         F.lit(run_id))
    .withColumn("ingestion_date", F.to_date(F.lit(ingestion_date)))
)

bronze.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_artists")
print(f"bronze_artists: {bronze.count()} rows written")